# Hente og lese inn data til pandas

## CSV 

* En vanlig måte å lagre data på er i csv-format
* csv = comma separated values
* I en csv-fil har vi data lagret som tekst i en type tabellformat
* Hver linje i filen er et datapunkt, og inneholder et eller flere felt med data (kolonner)
* Datafeltene er separert med en *separator*, ofte et komma
* Første linje i filen gir gjerne metadata (navn på kolonnene)

## Tegnkoding

* CSV-filer er som sagt vanlig tekst, men:
    - Tekst kan representeres på forkjellige måter i en datamaskin
    - Måten kalles tegnkodingen (character-coding)
    - Vi må ofte sørge for riktig inputkoding (input-encoding) for å få ut riktig tekst
* Enkleste mulige tegnkoding er ASCII og ANSI 
* Unicode sørger for at vi kan bruke æ,ø,å $\Delta$, $\Gamma$ osv. Feks UTF-8 og UTF-16

<img src="https://upload.wikimedia.org/wikipedia/commons/1/1b/ASCII-Table-wide.svg">

## ANSI
<img src="https://www.itwissen.info/lex-images/Zeichen-des-ANSI-Codes.png">

## Unicode
<img src="https://i.stack.imgur.com/6C0C6.png">

## CSV + Pandas

* Vi bruker pandas til å lese og lagre csv-filer
* `pd.read_csv("filnavn")`
* `read_csv` har **haugevis** med keyword arguments for å lese rare og potensielt føkka csv-filer
* Vi burde i de fleste tilfeller klare oss med:
    - `encoding = "input-enc"` feks `"utf-8"`
    - `sep = "separator"` feks `","` eller "`\t`" (tab)
    -  `header = rad` feks `header=0`dersom første rad gir kolonnenavnene

In [ ]:
import pandas as pd
#Last in studentdata fra blackboard
blackboard_df = pd.read_csv("blackboard.csv", sep="\t", encoding="utf-16") # For SSB encoding = "ISO-8859-1"
blackboard_df

* Filen vi har lastet inn er klasselisten fra blackboard
* på iirmoodle.it.ntnu.no er det mulig å melde folk opp i fag ved å laste *opp* en csv-fil
* `moodle_example.csv` viser hvordan denne filen skal se ut

In [ ]:
#Last inn eksempeldata fra moodle
moodleEx_df = pd.read_csv("moodle_example.csv")
moodleEx_df

### Opppgave 1

* Lag et dataframe fra "blackboard.csv" som er formatert slik moodle vil ha det

In [ ]:
email_addresser = [f"{brukernavn}@stud.ntnu.no" for brukernavn in blackboard_df["Brukernavn"]]

data = {"username": blackboard_df["Brukernavn"],
       "firstname": blackboard_df["Fornavn"],
       "lastname": blackboard_df["Etternavn"],
       "email": email_addresser }

moodleFormatert_df = pd.DataFrame(data)
moodleFormatert_df

* Vi lagrer et dataframe til csv med `df.to_csv("filnavn.csv", **kwargs)`
* når vi ser `**kwargs` på denne måten, betyr det at her kommer «keyword arguments»
* Vi kan se i [dokumentasjonen](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html) får å finne hvilke «kwargs» funksjonen tar

In [ ]:
#Lagre det nye datasettet 
moodleFormatert_df.to_csv("moodleformatert.csv", index = False)

* Det er mye å holde styr på i Pandas, og vi går ikke igjennom alle aspekter
* Ha en [cheat sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) for hånden
* Slå opp i diverse [tutorials](https://pandas.pydata.org/pandas-docs/stable/getting_started/tutorials.html)
* Spesielt [denne](https://www.skytowner.com/explore/pandas_recipes_reference) kan være kjekk (Pandas oppskrifter :) )

# Pandas i praksis

* Vi kan hente data å analysere, feks fra [statistisk sentralbyrå](http://www.ssb.no)
* SSB bruker tegnkodinger «UTF-8» og «ISO-8859-1»

In [ ]:
#Vi går til ssb.no og henter et datasett
arbeidsledige_df = pd.read_csv("08518_20231018-095706.csv", sep = "\t")


Dersom vi kun er interesserte i noen av verdiene kan vi bruke slicing. La oss si at vi er interesserte i verdiene for de ti siste årene.

In [ ]:
arbeidsledige_df_siste = arbeidsledige_df[-10*4:]

arbeidsledige_første_kvartal_df = arbeidsledige_df[::4]


In [ ]:
# Vi henter et datasett med åpnede konkurser fra SSB
konkurser_df = pd.read_csv("09695_20231018-104409.csv", sep = "\t", encoding = "ISO-8859-1")
konkurser_df

* Vi vil slå sammen de to datasettene
* Deretter vil se på sammenhengen mellom konkurser og arbeidsledighet

In [ ]:
konkurser_df+arbeidsledige_df #Det funket dårlig....

In [ ]:
 # Antall rader med data

# Vi summer sammen 3 og 3 rader -> kvartaler
n = konkurser_df.index.size
konkurser_kvartal = []
for i in range(0,n,3):
    konkurser_kvartal.append(konkurser_df.loc[i:i+2]["Opna konkursar"].sum()) 

#Vi lager ny indeks med format yyyyK# eks: 1994K3 = 3. kvartal 1994
index = []
for år in range(1980,2024):
    for kvartal in [1,2,3,4]:
        index.append(f"{år}K{kvartal}")

#Vi lager et nytt dataframe med konkurser pr. kvartal
konkurser_kvartal_df = pd.DataFrame({"Opna konkursar": konkurser_kvartal}, index = index[:len(konkurser_kvartal)])
konkurser_kvartal_df

* Vi kan legge til kolonnene fra et dataframe til et annet med `df1.join(df2)`
* Det finnes flere måter å slå sammen dataframes på, feks `append` eller `merge`
* Se [dokumentasjon](https://pandas.pydata.org/docs/user_guide/merging.html)

In [ ]:

#Slå sammen datasettene
joined_df = arbeidsledige_df.set_index("kvartal").join(konkurser_kvartal_df)
#df.set_index("kolonne") gjør verdier i oppgitt kolonne om til indeks
joined_df


* Å plotte fra et dataframe er herlig enkelt :)

In [ ]:
import matplotlib.pyplot as plt

#Plot en av kolonnene
joined_df["Arbeidsledige (1 000 personer)"].plot()
plt.title("Arbeidsledige per 1000 personer")
plt.show()
#Plot hele dataframe
joined_df.plot()
plt.title("Hele Dataframe")
plt.show()

#Vi bruker eller pyplot slik vi er vant med :)

* Å finne kovarians og korrelasjon er også lett

In [ ]:
joined_df.cov()

In [ ]:
joined_df.corr()

* De som trenger en oppfriskning på kovarians og korrelasjon kan se her:


### Kovarians
<a href="https://www.youtube.com/watch?v=9Y0Alg8huJk" 
  target="_blank"><img src="https://img.youtube.com/vi/9Y0Alg8huJk/0.jpg" 
alt="IMAGE ALT TEXT HERE" width="240" height="180" border="10" /></a>

### Korrelasjon
<a href="https://www.youtube.com/watch?v=WpZi02ulCvQ" 
  target="_blank"><img src="https://img.youtube.com/vi/WpZi02ulCvQ/0.jpg" 
alt="IMAGE ALT TEXT HERE" width="240" height="180" border="10" /></a>
